# Lab 5.2 &mdash; Build a StateGraph from scratch

**Time:** about 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; LangChain &amp; LangGraph**

### What you will do
- Write nodes that return only the keys they change
- Write the reducer that merges those changes into the state
- Write the engine: edges, a router and a cycle, in about 40 lines
- Add a step budget so the cycle always ends
- Run the same nodes on real LangGraph, and stream each step

> **How this lab works.** Fill in every `BLANK`, then run the **Self-check** cell under each
> section. It prints `[PASS]`, `[FAIL]` or `[TODO]` for each check. Graded cells never call the
> sandbox model, so your score does not depend on it. Cells marked **Run it for real** do call
> the model. If it is not reachable, they print how to fix it instead of crashing.

> **Why build it first?** LangGraph hides about 40 lines of engine. Once you have written
> them, the LangGraph code in the deck and in Lab 5.3 reads as plain Python.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, copy, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "aac-lab-5-2")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one check. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then run this cell again)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# The sandbox already has a model set up: nothing to install, no key to enter.
LLM_BASE_URL = os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
LLM_MODEL    = os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("No model is set up here. In a sandbox terminal run `env | grep LAB_LLM`.")
        print("If it prints nothing, tell your trainer. The graded cells still work.")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model that talks to the sandbox model."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One call to the model. Returns text, or an error string. Never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not set up -- the graded cells still work)")

In [ ]:
# ------------------------------------------------- AskOps: the same data and tools as Module 4
# Six runbooks and three open incidents. Nothing here is real, and nothing leaves this notebook.
RUNBOOKS = [{'id': 'RB-101',
  'title': 'Payments API returns 502 after deploy',
  'service': 'payments',
  'tags': ['deploy', 'rollback', '502', 'gateway'],
  'steps': ['Check the deploy pipeline for the last release',
            'Compare error rate before and after the release',
            'Roll back with the release tool if the error rate doubled',
            'Open an incident if rollback does not clear it']},
 {'id': 'RB-102',
  'title': 'Database connection pool exhausted',
  'service': 'payments',
  'tags': ['database', 'pool', 'timeout', 'connections'],
  'steps': ['Confirm the pool metric is at its maximum',
            'Find long-running queries and their owners',
            'Raise the pool size only as a temporary measure',
            'File a ticket for the query that held connections']},
 {'id': 'RB-201',
  'title': 'Login latency above 2 seconds',
  'service': 'auth',
  'tags': ['latency', 'login', 'cache', 'slow'],
  'steps': ['Check the token cache hit rate',
            'Warm the cache if a node restarted',
            'Scale the auth service if CPU is above 80 percent']},
 {'id': 'RB-202',
  'title': 'Certificate expiring within 7 days',
  'service': 'auth',
  'tags': ['tls', 'certificate', 'expiry'],
  'steps': ['List certificates expiring this week',
            'Request renewal from the PKI portal',
            'Deploy the renewed certificate and verify the chain']},
 {'id': 'RB-301',
  'title': 'Nightly batch job did not finish',
  'service': 'reporting',
  'tags': ['batch', 'job', 'timeout', 'retry'],
  'steps': ['Read the job log for the last completed step',
            'Re-run from the failed step, not from the start',
            'Tell report consumers the expected delay']},
 {'id': 'RB-302',
  'title': 'Disk usage above 90 percent on report nodes',
  'service': 'reporting',
  'tags': ['disk', 'storage', 'cleanup'],
  'steps': ['Find the largest directories',
            'Delete report archives older than 30 days',
            'Add a retention rule so it does not recur']}]

INCIDENTS_AT_START = [{'id': 'INC-9001',
  'title': 'Payments 502s after 14:00 release',
  'severity': 'high',
  'service': 'payments',
  'opened_at': '2026-09-14T14:07:00',
  'runbook_id': 'RB-101'},
 {'id': 'INC-9002',
  'title': 'Slow logins in the morning peak',
  'severity': 'medium',
  'service': 'auth',
  'opened_at': '2026-09-15T09:12:00',
  'runbook_id': 'RB-201'},
 {'id': 'INC-9003',
  'title': 'Batch report late for finance',
  'severity': 'low',
  'service': 'reporting',
  'opened_at': '2026-09-16T06:30:00',
  'runbook_id': None}]
INCIDENTS = copy.deepcopy(INCIDENTS_AT_START)

def reset_data():
    """Put the incident list back as it started. The checks call this, so they leave no trace."""
    INCIDENTS[:] = copy.deepcopy(INCIDENTS_AT_START)

# The four AskOps tools from labs/module-4-agent/agent.py, as plain functions.
def search_runbooks(query, service=None):
    words = {w for w in query.lower().split() if len(w) >= 3}
    hits = [{"id": r["id"], "title": r["title"]} for r in RUNBOOKS
            if service in (None, r["service"])
            and words & set(r["title"].lower().split() + r["tags"])]
    return json.dumps(hits[:3])

def get_runbook(runbook_id):
    for r in RUNBOOKS:
        if r["id"] == runbook_id:
            return json.dumps(r)
    return f"ERROR: no runbook {runbook_id}. Use search_runbooks first."

def list_incidents():
    return json.dumps(INCIDENTS)

def open_incident(title, severity, service, runbook_id=None):
    """The only tool that WRITES. It adds a record that other people see."""
    incident = {"id": f"INC-{9001 + len(INCIDENTS)}", "title": title,
                "severity": severity, "service": service, "runbook_id": runbook_id}
    INCIDENTS.insert(0, incident)
    return json.dumps(incident)

print(len(RUNBOOKS), "runbooks,", len(INCIDENTS), "open incidents")

## Concept

A LangGraph graph has four parts:

| Part | What it is |
|---|---|
| **State** | one dictionary that every node can read |
| **Node** | a function: the state goes in, only the **changed keys** come out |
| **Edge** | which node runs next, always |
| **Conditional edge** | which node runs next, decided by a **router** function that reads the state |

A **cycle** is an edge that points back to an earlier node. Here is the AskOps graph you build:

```
classify --runbook--> search --> check --good, or 3 tries--> answer
    |                   ^          |
    |                   +--weak----+
    +--incident--> read_incidents ---------------------------> answer
```

## Section 1 &mdash; Nodes return only what they change

This is the most important rule. A node returns **only the keys it changed**, and the engine merges
them into the state. That keeps each node small and easy to test on its own.

In [ ]:
QUESTION_502  = "Payments returns 502 after deploy"
QUESTION_DOWN = "The auth service is down for everyone"
QUESTION_ODD  = "Something feels odd today"

def fresh(question):
    """A new state for one question."""
    return {"question": question, "kind": None, "findings": [], "tries": 0,
            "steps": 0, "answer": None}

def classify(state: dict) -> dict:
    """Node: is this an outage, or a how-to-fix question?"""
    down = any(w in state["question"].lower() for w in ("down", "outage"))
    return {"kind": "incident" if down else "runbook", "steps": state["steps"] + 1}

def read_incidents(state: dict) -> dict:
    """Node: list the open incidents."""
    ids = [i["id"] for i in json.loads(list_incidents())]
    return {"findings": [f"incidents: {', '.join(ids)}"], "steps": state["steps"] + 1}

def search(state: dict) -> dict:
    """Node: search the runbooks for the question. Counts one more try."""
    hits = json.loads(search_runbooks(state["question"]))
    found = ", ".join(h["id"] for h in hits) or "nothing"
    return {"findings": [f"runbooks: {found}"],
            "tries": state["tries"] + 1,
            "steps": state["steps"] + 1}

In [ ]:
# --- Self-check: Section 1
_s0 = fresh(QUESTION_502)

check("a node returns only what it changed",
      lambda: set(search(_s0)) == {"findings", "tries", "steps"},
      "returning the whole state makes nodes hard to join together")
check("search records what it found", lambda: "RB-101" in search(_s0)["findings"][0])
check("search counts one more try", lambda: search(_s0)["tries"] == 1)
check("a question with no match says so",
      lambda: search(fresh(QUESTION_ODD))["findings"] == ["runbooks: nothing"])
check("classify sends an outage to the incidents path",
      lambda: classify(fresh(QUESTION_DOWN))["kind"] == "incident")
check("nodes do not change the state they were given",
      lambda: (search(_s0), _s0["tries"] == 0 and _s0["findings"] == [])[1],
      "return a new dict; never edit the state in place")

## Section 2 &mdash; Reducers: how a node's changes join the state

`tries` should be **replaced** by the new value. `findings` should **grow**: every node adds to the
list. That difference is the **reducer**. You declare it once for each key, instead of remembering
it in every node. In LangGraph, `add_messages` is the reducer for the message list.

In [ ]:
def replace(old, new):
    return new

def append(old, new):
    return list(old or []) + list(new)

REDUCERS = {
    "findings": append,             # every node's findings are kept, in order
    "kind": replace, "tries": replace, "steps": replace, "answer": replace, "question": replace,
}

def merge(state: dict, update: dict) -> dict:
    """Apply a node's changes, using the reducer declared for each key."""
    out = dict(state)
    for key, value in update.items():
        reducer = REDUCERS.get(key, replace)
        out[key] = reducer(state.get(key), value)
    return out

In [ ]:
# --- Self-check: Section 2
def _a():
    return merge(_s0, {"findings": ["one"], "tries": 1})

def _b():
    return merge(_a(), {"findings": ["two"], "tries": 2})

check("findings grow across nodes", lambda: _b()["findings"] == ["one", "two"],
      "this is the append reducer at work")
check("tries is replaced, not added to a list", lambda: _b()["tries"] == 2)
check("keys a node did not return stay the same", lambda: _b()["question"] == QUESTION_502)
check("the original state is not changed", lambda: _s0["findings"] == [])
check("a key with no reducer is replaced", lambda: merge(_s0, {"novel": 7})["novel"] == 7)
check("two writes to findings both survive",
      lambda: merge(merge(_s0, {"findings": ["x"]}), {"findings": ["y"]})["findings"] == ["x", "y"],
      "with replace, the first write would be lost, with no error")

## Section 3 &mdash; The engine: edges, a router and a cycle

The engine runs a node, merges its changes, and then picks the next node. A **plain edge** always
goes to the same node. A **conditional edge** calls a router. The cycle here is `check` sending a
weak search back to `search`. The engine also has a **step budget**, the same idea as `MAX_STEPS` on
Day 1, so a cycle can never run forever.

In [ ]:
END = "__end__"

class Graph:
    def __init__(self):
        self.nodes, self.edges, self.conditions = {}, {}, {}
        self.entry = None

    def add_node(self, name, fn):
        self.nodes[name] = fn
        return self

    def add_edge(self, src, dst):
        self.edges[src] = dst
        return self

    def add_conditional_edge(self, src, router):
        """router(state) returns the name of the next node, or END."""
        self.conditions[src] = router
        return self

    def set_entry(self, name):
        self.entry = name
        return self

    def run(self, state, max_steps=8):
        """Run until END, or until the step budget is spent. Returns (final_state, path)."""
        current, path = self.entry, []
        while current != END:
            if state["steps"] >= max_steps:
                return merge(state, {"answer": "Stopped: step budget spent."}), path
            path.append(current)
            state = merge(state, self.nodes[current](state))
            if current in self.conditions:
                current = self.conditions[current](state)
            else:
                current = self.edges.get(current, END)
        return state, path

In [ ]:
def check_node(state):
    """Node: nothing to change. The router after it makes the decision."""
    return {"steps": state["steps"] + 1}

def answer(state):
    runbooks = [f for f in state["findings"] if f.startswith("runbooks: RB")]
    if runbooks:
        text = "Follow " + runbooks[-1].split(": ")[1] + "."
    elif state["kind"] == "incident":
        text = "Check the open incidents first: " + state["findings"][-1]
    else:
        text = "No runbook found. Hand over to a person."
    return {"answer": text, "steps": state["steps"] + 1}

def route(state):
    """Router after classify: the kind decides the path."""
    return "search" if state["kind"] == "runbook" else "read_incidents"

def good_enough(state):
    """Router after check: stop when a runbook is found, or after 3 tries."""
    found = any(f.startswith("runbooks: RB") for f in state["findings"])
    return "answer" if found or state["tries"] >= 3 else "search"

def build():
    return (Graph()
            .add_node("classify", classify).add_node("search", search)
            .add_node("check", check_node).add_node("read_incidents", read_incidents)
            .add_node("answer", answer)
            .add_conditional_edge("classify", route)
            .add_edge("search", "check")
            .add_conditional_edge("check", good_enough)
            .add_edge("read_incidents", "answer")
            .add_edge("answer", END)
            .set_entry("classify"))

def _run(question, max_steps=12):
    return build().run(fresh(question), max_steps)

try:
    for q in (QUESTION_502, QUESTION_DOWN, QUESTION_ODD):
        final, path = _run(q)
        print(f"{q}\n  path  : {' -> '.join(path)}\n  answer: {final['answer']}\n")
except NameError:
    print("(fill in the blanks above, then run this cell again)")

In [ ]:
# --- Self-check: Section 3
check("a runbook question takes the search path",
      lambda: _run(QUESTION_502)[1] == ["classify", "search", "check", "answer"])
check("it answers with the runbook it found", lambda: "RB-101" in _run(QUESTION_502)[0]["answer"])
check("an outage takes the incidents path",
      lambda: _run(QUESTION_DOWN)[1] == ["classify", "read_incidents", "answer"])
check("a weak search goes round the cycle three times",
      lambda: _run(QUESTION_ODD)[1].count("search") == 3,
      "the router in check sends it back to search until tries reaches 3")
check("after three tries it hands over to a person",
      lambda: "Hand over" in _run(QUESTION_ODD)[0]["answer"])
check("findings from every search are kept", lambda: len(_run(QUESTION_ODD)[0]["findings"]) == 3)
check("a cycle whose router never says stop ends on the budget",
      lambda: "budget" in (Graph()
              .add_node("search", search)
              .add_conditional_edge("search", lambda s: "search")
              .set_entry("search")
              .run(fresh(QUESTION_ODD))[0]["answer"]),
      "a cycle with no budget is an endless loop")

## Run it for real

The same nodes and routers on real LangGraph. Very little changes. `TypedDict` describes the state.
`Annotated[list, add]` is your `append` reducer. `add_conditional_edges` takes your router. And
`recursion_limit` is your step budget.

`stream_mode="updates"` prints each node as it finishes, with only the keys it changed.

In [ ]:
try:
    from typing import Annotated
    from typing_extensions import TypedDict
    from operator import add
    from langgraph.graph import StateGraph, START, END as LG_END

    class AskState(TypedDict):
        question: str
        kind: str | None
        findings: Annotated[list, add]       # <- your append reducer, declared once
        tries: int
        steps: int
        answer: str | None

    g = StateGraph(AskState)
    for name, fn in [("classify", classify), ("search", search), ("check", check_node),
                     ("read_incidents", read_incidents), ("answer", answer)]:
        g.add_node(name, fn)
    g.add_edge(START, "classify")
    g.add_conditional_edges("classify", route, {"search": "search", "read_incidents": "read_incidents"})
    g.add_edge("search", "check")
    g.add_conditional_edges("check", good_enough, {"search": "search", "answer": "answer"})
    g.add_edge("read_incidents", "answer")
    g.add_edge("answer", LG_END)
    app = g.compile()

    for update in app.stream(fresh(QUESTION_ODD), {"recursion_limit": 20}, stream_mode="updates"):
        for node, changes in update.items():
            print(f"{node:<15} changed {sorted(changes)}")
    print("\nanswer:", app.invoke(fresh(QUESTION_502))["answer"])
except ImportError as exc:
    print(f"LangGraph is not installed here ({exc}). The graded cells above do not need it.")
except NameError:
    print("(fill in the blanks above, then run this cell again)")
except Exception as exc:
    print(f"<graph run failed: {type(exc).__name__}: {exc}>")

Now let the sandbox model write the answer. Only the `answer` node changes. The graph, the routers
and the reducer stay the same, so you can test everything except the model's words without a model.

In [ ]:
def model_answer(state):
    text = ask("Answer the on-call engineer in at most 3 lines, only from these findings. "
               "Cite runbook ids. If nothing was found, say so.\n\nQuestion: " + state["question"]
               + "\nFindings:\n" + "\n".join(state["findings"]))
    return {"answer": text, "steps": state["steps"] + 1}

if llm_ready():
    try:
        g2 = StateGraph(AskState)
        for name, fn in [("classify", classify), ("search", search), ("check", check_node),
                         ("read_incidents", read_incidents), ("answer", model_answer)]:
            g2.add_node(name, fn)
        g2.add_edge(START, "classify")
        g2.add_conditional_edges("classify", route, {"search": "search", "read_incidents": "read_incidents"})
        g2.add_edge("search", "check")
        g2.add_conditional_edges("check", good_enough, {"search": "search", "answer": "answer"})
        g2.add_edge("read_incidents", "answer")
        g2.add_edge("answer", LG_END)
        print(g2.compile().invoke(fresh(QUESTION_502))["answer"])
    except NameError:
        print("(fill in the blanks above, then run this cell again)")
    except Exception as exc:
        print(f"<graph run failed: {type(exc).__name__}: {exc}>")

### Read it

Your `merge` is LangGraph's reducer system. Your `conditions` are `add_conditional_edges`. Your
`max_steps` is `recursion_limit`. What LangGraph adds on top is the hard part: **saving the state
after every node**. That is Lab 5.3.

In [ ]:
score()

## Your turn

1. Make `search` and `read_incidents` both run after `classify`, for every question. Which reducer
   must `findings` have so that both results survive? Try it with `replace` and see what you lose.
2. Every node adds 1 to `steps` itself, so a node that forgets makes the budget wrong. Move the
   counting into `Graph.run`. What does a node lose, and what does it gain?